In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Dataset path
data_dir = "/content/drive/MyDrive/Colab Notebooks/CatDog"

categories = ["cats", "Dog"]

data = []
labels = []

# Load images
for category in categories:
    folder_path = os.path.join(data_dir, category)
    label = categories.index(category)

    for img in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img)

        image = cv2.imread(img_path)

        # Skip if image is not loaded
        if image is None:
            print("Skipping:", img_path)
            continue


        image = cv2.resize(image,(128,128))
        image = image / 255.0
        image = image.flatten()

        data.append(image)
        labels.append(label)

# Convert to numpy arrays
data = np.array(data)
labels = np.array(labels)

print("Total images loaded:", len(data))

# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    data, labels, test_size=0.2, random_state=42
)


Total images loaded: 4204


In [5]:
# Train Model (KNN)
model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)

# Test Accuracy
predictions = model.predict(X_test)
from sklearn.ensemble import RandomForestClassifier


print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.5505350772889417


In [6]:

# Train Model (KNN)
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
##model.fit(X_train, y_train)
# Test Accuracy
predictions = model.predict(X_test)
from sklearn.ensemble import RandomForestClassifier


print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.6587395957193817


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os
from PIL import Image

# =========================
# 1. FIX & CLEAN DATASET
# =========================
dataset_path = "/content/drive/MyDrive/Colab Notebooks/CatDog"

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)
        try:
            img = Image.open(path).convert("RGB")
            img.save(path)
        except:
            os.remove(path)

# =========================
# 2. LOAD DATASET
# =========================
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(128,128),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(128,128),
    batch_size=32
)

# =========================
# 3. OPTIMIZATION
# =========================
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# =========================
# 4. MODEL (CNN)
# =========================
model = models.Sequential([
    layers.Input(shape=(128,128,3)),

    layers.Rescaling(1./255),

    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),

    layers.Conv2D(32,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,3,activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

# =========================
# 5. COMPILE
# =========================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# =========================
# 6. TRAIN
# =========================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop]
)

# =========================
# 7. EVALUATE
# =========================
loss, acc = model.evaluate(val_ds)
print("Final Accuracy:", acc)

Found 4204 files belonging to 2 classes.
Using 3364 files for training.
Found 4204 files belonging to 2 classes.
Using 840 files for validation.
Epoch 1/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.5095 - loss: 0.7103 - val_accuracy: 0.5274 - val_loss: 0.6863
Epoch 2/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 178s 1s/step - accuracy: 0.5571 - loss: 0.6853 - val_accuracy: 0.5929 - val_loss: 0.6716
Epoch 3/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 138s 1s/step - accuracy: 0.5719 - loss: 0.6780 - val_accuracy: 0.6214 - val_loss: 0.6550
Epoch 4/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.5936 - loss: 0.6742 - val_accuracy: 0.5774 - val_loss: 0.6778
Epoch 5/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - accuracy: 0.5972 - loss: 0.6723 - val_accuracy: 0.6060 - val_loss: 0.6514
Epoch 6/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 130s 1s/step - accuracy: 0.6156 - loss: 0.6594 - val_accuracy: 0.6107 - val_loss: 0.6924
Epoch 7/15
106/106 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step - accuracy: 0.6249 - loss: 0

In [ ]:
import os
from PIL import Image

dataset_path = "/content/drive/MyDrive/Colab Notebooks/CatDog"

bad_images = 0

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        path = os.path.join(root, file)
        try:
            img = Image.open(path)
            img = img.convert("RGB")   # force RGB
        except:
            print("Deleting:", path)
            os.remove(path)
            bad_images += 1

print("Removed images:", bad_images)

Removed images: 0
